In [345]:
from models.gpkg import GeoPackage
import numpy as np

In [394]:
gpkg = GeoPackage()
ti = gpkg.read_layer('ti_all_revisada_v1')
ti['ti_id'] = ti.index
pop = gpkg.read_layer('pop_indigena_mun')
bioma = gpkg.read_layer('biomas_rs_faixa_trans_final')
pop['mun_id'] = pop.index

In [395]:
pop['PessInd'] = pop['PessInd'].fillna(0).astype(int)
pop['pop_indigena'] = pop['PessInd'].mask(pop['PessInd'] < 0)
pop['PopIndEmT']= pop['PessIndEmT'].fillna(0).astype(int)
pop['pop_indigena_ti'] = pop['PessIndEmT'].mask(pop['PessIndEmT'] < 0)

In [396]:
pop['PopResid'] = pop['PopResid'].fillna(0).astype(int)
pop['pop_total'] = pop['PopResid'].mask(pop['PopResid'] < 0)


In [397]:
pop['perc_indigena'] = pop['pop_indigena'] / pop['pop_total']
pop['perc_em_ti'] = pop['pop_indigena_ti'] / pop['pop_indigena']
pop['area_mun'] = pop.geometry.area / 1e6

In [402]:
from tools.exposure import spatial_join_pampa

In [403]:
merge = spatial_join_pampa(pop, bioma, 'mun_id')

In [409]:
merge.groupby('bioma').size()

bioma
Faixa de transição    166
Mata atlântica        202
Pampa                 129
dtype: int64

In [408]:
merge.groupby('bioma')[['pop_indigena', 'pop_indigena_ti']].sum()

,pop_indigena,pop_indigena_ti
bioma,,
Faixa de transição,15396,9372.0
Mata atlântica,10445,5843.0
Pampa,10261,415.0


In [350]:
pop['Nome'].loc[pop['perc_em_ti'] == pop['perc_em_ti'].min()]

168    Faxinalzinho - RS
Name: Nome, dtype: object

In [242]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

# intersect partes
parts = gpd.overlay(ti[['ti_id','geometry']], pop[['mun_id','geometry']], how='intersection')
parts['area_km2'] = parts.geometry.area / 1e6

# soma areas por mun
areasum = parts.groupby('mun_id', as_index=False).area_km2.sum().rename(columns={'area_km2':'areasum_mun'})
parts = parts.merge(areasum, on='mun_id', how='left')
parts = parts.merge(pop[['mun_id','pop_total','pop_indigena','pop_indigena_ti']], on='mun_id', how='left')

# 1️⃣ Alocação principal: usar pop_indigena_ti quando disponível
parts['pop_alloc'] = np.where(
    parts['pop_indigena_ti'].notna() & (parts['areasum_mun'] > 0),
    parts['pop_indigena_ti'] * (parts['area_km2'] / parts['areasum_mun']),
    np.nan
)

# 2️⃣ Fallback: se pop_indigena_ti é NaN, usar pop_indigena proporcional à área
parts['pop_alloc_fallback'] = np.where(
    parts['pop_indigena_ti'].isna() & parts['pop_indigena'].notna() & (parts['areasum_mun'] > 0),
    parts['pop_indigena'] * (parts['area_km2'] / parts['areasum_mun']),
    np.nan
)

# 3️⃣ Combinar alocação principal + fallback
parts['pop_part_final'] = parts['pop_alloc'].combine_first(parts['pop_alloc_fallback'])

# 4️⃣ Calcular IR_municipal com fallback robusto
parts['IR_municipal_corrigido'] = np.where(
    parts['pop_indigena_ti'].notna() & (parts['pop_indigena_ti'] > 0),
    parts['pop_part_final'] / parts['pop_indigena_ti'],  # usar dado da TI quando disponível
    np.where(
        parts['pop_indigena'].notna() & (parts['pop_indigena'] > 0),
        parts['pop_part_final'] / parts['pop_indigena'],   # fallback: população indígena do município
        0
    )
)

# 5️⃣ Calcular IR_global (normalização pelo máximo da população estimada)
ti_max_pop = parts.groupby('ti_id')['pop_part_final'].sum().max()
ti_grouped = parts.groupby('ti_id', as_index=False).agg({
    'pop_part_final': 'sum'
})
ti_grouped['IR_global'] = ti_grouped['pop_part_final'] / ti_max_pop

# 6️⃣ Merge IR_municipal médio por TI (caso TI cruza múltiplos municípios)
IR_mun_avg = parts.groupby('ti_id')['IR_municipal_corrigido'].mean().reset_index()
ti_grouped = ti_grouped.merge(IR_mun_avg, on='ti_id', how='left')
ti_grouped.rename(columns={'IR_municipal_corrigido': 'IR_municipal_corrigido_avg'}, inplace=True)

# 7️⃣ NOVO: Verificação de sobreposição de municípios com múltiplas TIs
mun_with_multiple_tis = parts.groupby('mun_id')['ti_id'].nunique().reset_index()
mun_with_multiple_tis.rename(columns={'ti_id': 'n_tis'}, inplace=True)
mun_with_multiple_tis = mun_with_multiple_tis[mun_with_multiple_tis['n_tis'] > 1]
print(f"Número de municípios com múltiplas TIs: {len(mun_with_multiple_tis)}")

# 8️⃣ NOVO: Análise de sensibilidade - abordagem alternativa baseada em densidade
parts['pop_density_alloc'] = np.where(
    parts['pop_indigena'].notna() & (parts['areasum_mun'] > 0),
    (parts['pop_indigena'] / parts['areasum_mun']) * parts['area_km2'],
    np.nan
)

# 9️⃣ NOVO: Calcular erro de estimativa quando ambos os dados estão disponíveis
parts['error_estimate'] = np.where(
    parts['pop_indigena_ti'].notna() & parts['pop_alloc_fallback'].notna(),
    abs(parts['pop_alloc'] - parts['pop_alloc_fallback']) / parts['pop_indigena_ti'],
    np.nan
)

print(f"Erro médio da estimativa fallback: {parts['error_estimate'].mean():.4f}")

# 🔟 NOVO: Validação dos resultados
total_pop_ti = parts.groupby('ti_id')['pop_part_final'].sum().sum()
total_pop_mun = parts['pop_indigena'].sum()

print(f"População total estimada nas TIs: {total_pop_ti:.0f}")
print(f"População indígena municipal total: {total_pop_mun:.0f}")
print(f"Razão entre as estimativas: {total_pop_ti/total_pop_mun:.4f}")

# 1️⃣1️⃣ NOVO: Salvar resultados intermediários para análise
parts.to_csv('detalhamento_alocacao_populacional.csv', index=False)
print("Arquivo salvo: detalhamento_alocacao_populacional.csv")

# 1️⃣2️⃣ NOVO: Análise de distribuição dos índices de representatividade
plt.figure(figsize=(10, 6))
plt.hist(ti_grouped['IR_global'], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
plt.title('Distribuição do Índice de Representatividade Global (IR_global)')
plt.xlabel('IR_global')
plt.ylabel('Frequência')
plt.grid(axis='y', alpha=0.75)
plt.savefig('distribuicao_IR_global.png')
plt.close()

# 1️⃣3️⃣ Resultado final pronto para análise de vulnerabilidade
ti_grouped.to_csv('ti_representatividade_final.csv', index=False)
print("Arquivo salvo: ti_representatividade_final.csv")

# 1️⃣4️⃣ NOVO: Relatório de qualidade dos dados
relatorio_qualidade = {
    'total_tis': ti_grouped['ti_id'].nunique(),
    'total_municipios': parts['mun_id'].nunique(),
    'tis_com_dados_diretos': parts[parts['pop_indigena_ti'].notna()]['ti_id'].nunique(),
    'media_IR_global': ti_grouped['IR_global'].mean(),
    'media_IR_municipal': ti_grouped['IR_municipal_corrigido_avg'].mean(),
    'erro_medio_estimativa': parts['error_estimate'].mean(),
    'municipios_multiplas_tis': len(mun_with_multiple_tis)
}

print("\n=== RELATÓRIO DE QUALIDADE ===")
for k, v in relatorio_qualidade.items():
    print(f"{k}: {v}")

Número de municípios com múltiplas TIs: 49
Erro médio da estimativa fallback: nan
População total estimada nas TIs: 29681
População indígena municipal total: 101988
Razão entre as estimativas: 0.2910
Arquivo salvo: detalhamento_alocacao_populacional.csv
Arquivo salvo: ti_representatividade_final.csv

=== RELATÓRIO DE QUALIDADE ===
total_tis: 141
total_municipios: 121
tis_com_dados_diretos: 42
media_IR_global: 0.03850431720805507
media_IR_municipal: 0.47586200880860946
erro_medio_estimativa: nan
municipios_multiplas_tis: 49


In [351]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# intersect partes
parts = gpd.overlay(ti[['ti_id','geometry']], pop[['mun_id','geometry', 'area_mun']], how='intersection')
parts['area_km2'] = parts.geometry.area / 1e6

# soma areas por mun
areasum = parts.groupby('mun_id', as_index=False).area_km2.sum().rename(columns={'area_km2':'areasum_mun'})
parts = parts.merge(areasum, on='mun_id', how='left')
parts = parts.merge(pop[['mun_id','pop_total','pop_indigena','pop_indigena_ti']], on='mun_id', how='left')

# 1️⃣ VERIFICAR a estrutura dos dados
print(f"Quantidade de TIs: {parts['ti_id'].nunique()}")
print(f"Quantidade de municípios no parts: {parts['mun_id'].nunique()}")
print(f"Quantidade de intersecções (parts): {len(parts)}")

# 2️⃣ CORREÇÃO: Usar pop_indigena (população indígena MUNICIPAL) para o cálculo
pop_total_municipal = parts[['mun_id', 'pop_indigena']].drop_duplicates()['pop_indigena'].sum()
pop_total_ti = parts[['mun_id', 'pop_indigena_ti']].drop_duplicates()['pop_indigena_ti'].sum()

fator_correcao_mun = pop_total_municipal / parts[['mun_id', 'pop_indigena']].drop_duplicates()['pop_indigena'].sum()
print(f"Fator de correção municipal: {fator_correcao_mun:.4f}")


print(f"População indígena MUNICIPAL total: {pop_total_municipal}")
print(f"População indígena DENTRO das TIs: {pop_total_ti}")
# PRIMEIRO: Garantir que temos a população oficial correta
# Vamos carregar os dados oficiais do Censo

pop_censo = pd.DataFrame({
    'mun_id': parts['mun_id'].unique(),
    'pop_indigena_censo': pop.pop_indigena_ti.astype('Int64')[pop.mun_id.isin(parts['mun_id'].unique())]  # Este valor deve vir do seu dataframe oficial do Censo
})

# Se você tem o dataframe completo do Censo, use:
# pop_censo = pop[['mun_id', 'pop_indigena']].copy()
# pop_censo.rename(columns={'pop_indigena': 'pop_indigena_censo'}, inplace=True)

# 1️⃣ VERIFICAR a estrutura dos dados
print(f"Quantidade de TIs: {parts['ti_id'].nunique()}")
print(f"Quantidade de municípios no parts: {parts['mun_id'].nunique()}")
print(f"Quantidade de intersecções (parts): {len(parts)}")

# 2️⃣ Calcular a população total atual nos parts
pop_total_parts = parts[['mun_id', 'pop_indigena_ti']].drop_duplicates()['pop_indigena_ti'].sum()
print(f"População total nos parts (com duplicação): {pop_total_parts}")
print(f"População oficial do Censo: {pop_censo['pop_indigena_censo'].sum()}")

# 3️⃣ CORREÇÃO: Ajustar a população municipal para bater com o Censo
# Primeiro, vamos ver quais municípios estão no parts
muns_in_parts = parts['mun_id'].unique()

# Calcular fator de correção
fator_correcao = pop_censo['pop_indigena_censo'].sum() / pop_total_parts
print(f"Fator de correção necessário: {fator_correcao:.4f}")

# 4️⃣ Aplicar o fator de correção à população municipal
parts['pop_indigena_corrigida'] = parts['pop_indigena_ti'] * fator_correcao

# 5️⃣ Calcular a área total de cada município que intersecta TIs
mun_area = parts.groupby('mun_id')['area_km2'].sum().reset_index()
mun_area.rename(columns={'area_km2': 'area_total_mun_tis'}, inplace=True)

parts = parts.merge(mun_area, on='mun_id')

# 6️⃣ Alocação principal: usar pop_indigena_ti quando disponível
parts['pop_alloc'] = np.where(
    parts['pop_indigena_ti'].notna() & (parts['areasum_mun'] > 0),
    parts['pop_indigena_ti'] * (parts['area_km2'] / parts['areasum_mun']),
    np.nan
)

# 7️⃣ Fallback: se pop_indigena_ti é NaN, usar pop_indigena CORRIGIDA proporcional à área
parts['pop_alloc_fallback'] = np.where(
    parts['pop_indigena_ti'].isna() & (parts['area_total_mun_tis'] > 0),
    parts['pop_indigena_corrigida'] * (parts['area_km2'] / parts['area_total_mun_tis']),
    np.nan
)

# 8️⃣ Combinar alocação principal + fallback
parts['pop_part_final'] = parts['pop_alloc'].combine_first(parts['pop_alloc_fallback'])

# 9️⃣ Calcular IR_municipal com fallback robusto
parts['IR_municipal_corrigido'] = np.where(
    parts['pop_indigena_ti'].notna() & (parts['pop_indigena_ti'] > 0),
    parts['pop_part_final'] / parts['pop_indigena_ti'],
    np.where(
        parts['pop_indigena_corrigida'].notna() & (parts['pop_indigena_corrigida'] > 0),
        parts['pop_part_final'] / parts['pop_indigena_corrigida'],
        0
    )
)

# 🔟 Calcular IR_global (normalização pelo máximo da população estimada)
ti_max_pop = parts.groupby('ti_id')['pop_part_final'].sum().max()
ti_grouped = parts.groupby('ti_id', as_index=False).agg({
    'pop_part_final': 'sum'
})
ti_grouped['IR_global'] = ti_grouped['pop_part_final'] / ti_max_pop

# 1️⃣1️⃣ Merge IR_municipal médio por TI
IR_mun_avg = parts.groupby('ti_id')['IR_municipal_corrigido'].mean().reset_index()
ti_grouped = ti_grouped.merge(IR_mun_avg, on='ti_id', how='left')
ti_grouped.rename(columns={'IR_municipal_corrigido': 'IR_municipal_corrigido_avg'}, inplace=True)

# 1️⃣2️⃣ VERIFICAÇÃO FINAL
total_pop_ti_estimada = ti_grouped['pop_part_final'].sum()
total_pop_mun_corrigida = parts[['mun_id', 'pop_indigena_corrigida']].drop_duplicates()['pop_indigena_corrigida'].sum()

print(f"\n=== VERIFICAÇÃO DA CORREÇÃO ===")
print(f"População indígena municipal corrigida: {total_pop_mun_corrigida:.0f}")
print(f"População oficial do Censo: {pop_censo['pop_indigena_censo'].sum()}")
print(f"População total estimada nas TIs: {total_pop_ti_estimada:.0f}")
print(f"Razão entre estimativa/real: {total_pop_ti_estimada/total_pop_mun_corrigida:.4f}")

# 1️⃣3️⃣ Análise de distribuição
print(f"\n=== DISTRIBUIÇÃO DA POPULAÇÃO ===")
print(f"TIs com população estimada: {len(ti_grouped)}")
print(f"População média por TI: {ti_grouped['pop_part_final'].mean():.1f}")
print(f"Maior TI: {ti_grouped['pop_part_final'].max():.0f} habitantes")
print(f"Menor TI: {ti_grouped['pop_part_final'].min():.0f} habitantes")

# 1️⃣4️⃣ Salvar resultados
parts.to_csv('detalhamento_alocacao_populacional_corrigido.csv', index=False)
ti_grouped.to_csv('ti_representatividade_final_corrigido.csv', index=False)

print("\nArquivos salvos:")
print("- detalhamento_alocacao_populacional_corrigido.csv")
print("- ti_representatividade_final_corrigido.csv")

# 1️⃣5️⃣ Relatório final
print(f"\n=== RELATÓRIO FINAL ===")
print(f"TIs analisadas: {ti_grouped['ti_id'].nunique()}")
print(f"Municípios com TIs: {parts['mun_id'].nunique()}")
print(f"População municipal oficial: {pop_censo['pop_indigena_censo'].sum()}")
print(f"População municipal corrigida: {total_pop_mun_corrigida:.0f}")
print(f"População estimada nas TIs: {total_pop_ti_estimada:.0f}")
print(f"TIs com dados diretos: {parts[parts['pop_indigena_ti'].notna()]['ti_id'].nunique()}")

Quantidade de TIs: 141
Quantidade de municípios no parts: 121
Quantidade de intersecções (parts): 224
Fator de correção municipal: 1.0000
População indígena MUNICIPAL total: 32635
População indígena DENTRO das TIs: 15630.0
Quantidade de TIs: 141
Quantidade de municípios no parts: 121
Quantidade de intersecções (parts): 224
População total nos parts (com duplicação): 15630.0
População oficial do Censo: 15630
Fator de correção necessário: 1.0000

=== VERIFICAÇÃO DA CORREÇÃO ===
População indígena municipal corrigida: 15630
População oficial do Censo: 15630
População total estimada nas TIs: 15630
Razão entre estimativa/real: 1.0000

=== DISTRIBUIÇÃO DA POPULAÇÃO ===
TIs com população estimada: 141
População média por TI: 110.9
Maior TI: 5464 habitantes
Menor TI: 0 habitantes

Arquivos salvos:
- detalhamento_alocacao_populacional_corrigido.csv
- ti_representatividade_final_corrigido.csv

=== RELATÓRIO FINAL ===
TIs analisadas: 141
Municípios com TIs: 121
População municipal oficial: 15630


In [352]:
del parts

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd

# intersect partes
parts = gpd.overlay(ti[['ti_id','geometry']], pop[['mun_id','geometry', 'area_mun']], how='intersection')
parts['area_km2'] = parts.geometry.area / 1e6

# soma areas por mun
areasum = parts.groupby('mun_id', as_index=False).area_km2.sum().rename(columns={'area_km2':'areasum_mun'})
parts = parts.merge(areasum, on='mun_id', how='left')

# 🔴 CORREÇÃO: Usar as variáveis corretas do dataframe pop
parts = parts.merge(pop[['mun_id','pop_total','pop_indigena','pop_indigena_ti']], on='mun_id', how='left')

# 1️⃣ VERIFICAR a estrutura dos dados
print(f"Quantidade de TIs: {parts['ti_id'].nunique()}")
print(f"Quantidade de municípios no parts: {parts['mun_id'].nunique()}")
print(f"Quantidade de intersecções (parts): {len(parts)}")

# 2️⃣ CORREÇÃO: Usar pop_indigena (população indígena MUNICIPAL) para o cálculo
pop_total_municipal = parts[['mun_id', 'pop_indigena']].drop_duplicates()['pop_indigena'].sum()
pop_total_ti = parts[['mun_id', 'pop_indigena_ti']].drop_duplicates()['pop_indigena_ti'].sum()

print(f"População indígena MUNICIPAL total: {pop_total_municipal}")
print(f"População indígena DENTRO das TIs: {pop_total_ti}")

# 3️⃣ CORREÇÃO: Calcular fator de correção para população MUNICIPAL
# (se necessário, dependendo da qualidade dos dados)
fator_correcao_mun = pop_total_municipal / parts[['mun_id', 'pop_indigena']].drop_duplicates()['pop_indigena'].sum()
print(f"Fator de correção municipal: {fator_correcao_mun:.4f}")

parts['pop_indigena_corrigida'] = parts['pop_indigena'] * fator_correcao_mun

# 4️⃣ Calcular a área total de cada município que intersecta TIs
parts = parts.merge(areasum, on='mun_id')

# 5️⃣ Alocação principal: usar pop_indigena_ti quando disponível (dados DIRETOS da TI)
parts['pop_alloc'] = np.where(
    parts['pop_indigena_ti'].notna() & (parts['areasum_mun'] > 0),
    parts['pop_indigena_ti'] * (parts['area_km2'] / parts['areasum_mun']),
    np.nan
)
parts['densidade'] = parts['pop_indigena'] / parts['area_km2']
# 6️⃣ Fallback: se pop_indigena_ti é NaN, usar população MUNICIPAL proporcional à área
parts['pop_alloc_fallback'] = np.where(
    parts['pop_indigena_ti'].isna() & (parts['areasum_mun'] > 0),
    parts['pop_indigena_corrigida'] * (parts['area_km2'] / parts['areasum_mun']),
    np.nan
)

# 7️⃣ Combinar alocação principal + fallback
parts['pop_part_final'] = parts['pop_alloc'].combine_first(parts['pop_alloc_fallback'])

# 8️⃣ VERIFICAÇÃO: Comparar com dado real quando disponível
total_estimado_ti = parts['pop_part_final'].sum()
# total_real_ti = parts[parts['pop_indigena_ti'].notna()]['pop_indigena_ti'].sum()
total_real_ti = pop_total_ti

print(f"\n=== VERIFICAÇÃO ===")
print(f"População real em TIs com dados: {total_real_ti}")
print(f"População estimada total em TIs: {total_estimado_ti:.0f}")

# 9️⃣ Calcular fator de correção FINAL baseado na superestimativa
if total_real_ti > 0:
    fator_correcao_final = total_real_ti / total_estimado_ti
    print(f"Fator de correção final: {fator_correcao_final:.4f}")
    parts['pop_part_final_corrigido'] = parts['pop_part_final'] * fator_correcao_final
else:
    parts['pop_part_final_corrigido'] = parts['pop_part_final']

# 🔟 Calcular IR_municipal com fallback robusto
parts['IR_municipal_corrigido'] = np.where(
    parts['pop_indigena_ti'].notna() & (parts['pop_indigena_ti'] > 0),
    parts['pop_part_final_corrigido'] / parts['pop_indigena_ti'],
    np.where(
        parts['pop_indigena_corrigida'].notna() & (parts['pop_indigena_corrigida'] > 0),
        parts['pop_part_final_corrigido'] / parts['pop_indigena_corrigida'],
        0
    )
)

# 1️⃣1️⃣ Agrupar por TI para resultados finais
ti_grouped = parts.groupby('ti_id', as_index=False).agg({
    'pop_part_final_corrigido': 'sum',
    'area_km2': 'sum'
})

# Calcular IR_global
ti_grouped['pop_part_final_corrigido'] = ti_grouped['pop_part_final_corrigido'].astype(int)
ti_max_pop = ti_grouped['pop_part_final_corrigido'].max()
ti_grouped['IR_global'] = ti_grouped['pop_part_final_corrigido'] / ti_max_pop

# 1️⃣2️⃣ Resultados finais
print(f"\n=== RESULTADOS FINAIS ===")
print(f"População total estimada nas TIs: {ti_grouped['pop_part_final_corrigido'].sum():.0f}")
print(f"População real conhecida nas TIs: {total_real_ti}")
print(f"TIs com população estimada: {len(ti_grouped)}")
print(f"TIs com dados diretos: {parts[parts['pop_indigena_ti'].notna()]['ti_id'].nunique()}")

# Salvar resultados
parts.to_csv('detalhamento_alocacao_populacional_corrigido.csv', index=False)
ti_grouped.to_csv('ti_representatividade_final_corrigido.csv', index=False)

Quantidade de TIs: 141
Quantidade de municípios no parts: 121
Quantidade de intersecções (parts): 224
População indígena MUNICIPAL total: 32635
População indígena DENTRO das TIs: 15630.0
Fator de correção municipal: 1.0000

=== VERIFICAÇÃO ===
População real em TIs com dados: 15630.0
População estimada total em TIs: 29681
Fator de correção final: 0.5266

=== RESULTADOS FINAIS ===
População total estimada nas TIs: 15564
População real conhecida nas TIs: 15630.0
TIs com população estimada: 141
TIs com dados diretos: 42


In [392]:
dados_pop = {
    "População Total": [pop.pop_total.astype(int).sum()],
    "População Indígena Total": [pop.pop_indigena.astype(int).sum()],
    "População Indígena em TI Total": [pop.pop_indigena_ti.astype('Int64').sum()],
    "Quantidade de municípios amostra": [parts['mun_id'].nunique()],
    "Quantidade de municípios amostra com dados de pop em TI": [parts[parts['pop_indigena_ti'].notna()]['ti_id'].nunique()],
    "População Indígena total da amostra de municípios": [pop_total_municipal],
    "População Indígena em TI estimada": [ti_grouped['pop_part_final_corrigido'].astype(int).sum()]
}

In [393]:
pd.DataFrame.from_dict(dados_pop, orient='index').to_csv('data/dados_pop_estimada.csv')

In [382]:
print(mun_area.head())
print(areasum.head())

   mun_id  area_total_mun_tis
0       4        2.706391e+02
1       5        8.634584e+00
2      12        2.697796e-01
3      15        1.461626e-03
4      16        3.468185e-08
   mun_id   areasum_mun
0       4  2.706391e+02
1       5  8.634584e+00
2      12  2.697796e-01
3      15  1.461626e-03
4      16  3.468185e-08


In [376]:
mun_with_multiple_tis

,mun_id,n_tis
1,5,2
10,41,10
11,45,3
12,46,2
16,64,4
18,66,2
19,69,5
21,81,3
23,89,2
25,92,2


In [366]:
parts.area_total_mun_tis

0       20.055441
1       10.334302
2       26.363595
3       10.334302
4      174.538833
          ...    
219      3.355205
220      0.310339
221      1.827912
222      0.861514
223      0.962703
Name: area_total_mun_tis, Length: 224, dtype: float64

In [367]:
parts['densidade'] = parts['pop_indigena'] / parts.area_total_mun_tis

In [358]:
parts['area_km2']

0      2.973008e+00
1      3.295298e+00
2      3.147687e-08
3      1.460147e-01
4      1.131268e+00
           ...     
219    3.355205e+00
220    3.103395e-01
221    1.827912e+00
222    8.615140e-01
223    9.627027e-01
Name: area_km2, Length: 224, dtype: float64

In [368]:
# Alocação baseada na densidade municipal
parts['pop_alloc_fallback'] = np.where(
    parts['pop_indigena_ti'].isna() & (parts['area_total_mun_tis'] > 0),
    parts['densidade'] * parts['area_km2'],
    np.nan
)

In [371]:
parts['pop_alloc_fallback_2'] = np.where(
    parts['pop_indigena_ti'].isna() & (parts['area_total_mun_tis'] > 0),
    parts['pop_indigena_corrigida'] * (parts['area_km2'] / parts['area_total_mun_tis']),
    np.nan
)


In [373]:
parts['pop_alloc_fallback_2'].describe()

count     168.000000
mean       83.636905
std       195.203778
min         0.000000
25%         1.938670
50%        19.832987
75%        70.849716
max      1493.673018
Name: pop_alloc_fallback_2, dtype: float64

In [374]:
parts['pop_alloc_fallback'].describe()

count     168.000000
mean       83.636905
std       195.203778
min         0.000000
25%         1.938670
50%        19.832987
75%        70.849716
max      1493.673018
Name: pop_alloc_fallback, dtype: float64

In [417]:
# Calcular IR_municipal_corrigido médio por TI (usando média ponderada por área ou população)
IR_mun_corrigido_agg = parts.groupby('ti_id').apply(
    lambda x: np.average(x['IR_municipal_corrigido'], weights=x['area_km2'])
).reset_index(name='IR_municipal_corrigido_avg')

# Alternativa: média simples (caso prefira)
# IR_mun_corrigido_agg = parts.groupby('ti_id')['IR_municipal_corrigido'].mean().reset_index()

# Merge com o ti_grouped
ti_grouped = ti_grouped.merge(IR_mun_corrigido_agg, on='ti_id', how='left')

# Verificar resultados
print(ti_grouped[['ti_id', 'pop_part_final_corrigido', 'IR_global', 'IR_municipal_corrigido_avg']].head())
print(f"\nEstatísticas do IR Municipal Corrigido:")
print(ti_grouped['IR_municipal_corrigido_avg'].describe())

   ti_id  pop_part_final_corrigido  IR_global  IR_municipal_corrigido_avg
0      0                         8   0.002780                    0.078063
1      1                       496   0.172342                    0.167917
2      2                        22   0.007644                    0.007440
3      3                         0   0.000000                    0.003413
4      4                         1   0.000347                    0.014620

Estatísticas do IR Municipal Corrigido:
count    141.000000
mean       0.272323
std        0.218543
min        0.000170
25%        0.023189
50%        0.276171
75%        0.526224
max        0.526600
Name: IR_municipal_corrigido_avg, dtype: float64


C:\Users\adminitsd\AppData\Local\Temp\ipykernel_26400\3323528916.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  IR_mun_corrigido_agg = parts.groupby('ti_id').apply(


In [337]:
parts['pop_alloc_fallback']

0       16.009861
1      942.898411
2             NaN
3       41.779829
4        1.704626
          ...    
219     28.000000
220      9.000000
221     26.000000
222     18.000000
223     91.000000
Name: pop_alloc_fallback, Length: 224, dtype: float64

In [420]:
ti_grouped

,ti_id,pop_part_final_corrigido,area_km2,IR_global,IR_municipal_corrigido_avg
0,0,8,2.973008,0.002780,0.078063
1,1,496,3.295298,0.172342,0.167917
2,2,22,0.146015,0.007644,0.007440
3,3,0,1.131268,0.000000,0.003413
4,4,1,0.556802,0.000347,0.014620
...,...,...,...,...,...
136,136,14,3.355205,0.004864,0.526600
137,137,4,0.310339,0.001390,0.526600
138,138,13,1.827912,0.004517,0.526600
139,139,9,0.861514,0.003127,0.526600


In [291]:
[pop.mun_id.isin( parts['mun_id'].unique())]

[0      False
 1      False
 2      False
 3      False
 4       True
        ...  
 492    False
 493    False
 494     True
 495    False
 496    False
 Name: mun_id, Length: 497, dtype: bool]

In [278]:
parts['mun_id'].drop_duplicates().sum()

27990

In [418]:
import pandas as pd
df = pd.merge(ti, ti_grouped, how='left', on='ti_id')

In [419]:
merge_df = spatial_join_pampa(df, bioma, 'ti_id')

In [421]:
merge_df.groupby('bioma')[['pop_part_final_corrigido', 'IR_municipal_corrigido_avg']].sum()

,pop_part_final_corrigido,IR_municipal_corrigido_avg
bioma,,
Faixa de transição,8951,13.719180
Mata atlântica,3391,11.719001
Pampa,3222,12.959363


In [229]:
import geopandas as gpd
import pandas as pd
import numpy as np

# intersect partes
parts = gpd.overlay(ti[['ti_id','geometry']], pop[['mun_id','geometry']], how='intersection')
parts['area_km2'] = parts.geometry.area / 1e6

# soma areas por mun
areasum = parts.groupby('mun_id', as_index=False).area_km2.sum().rename(columns={'area_km2':'areasum_mun'})
parts = parts.merge(areasum, on='mun_id', how='left')
parts = parts.merge(pop[['mun_id','pop_total','pop_indigena','pop_indigena_ti']], on='mun_id', how='left')

# alocação area-weighted quando houver pop_TI_mun
parts['pop_alloc'] = np.where(
    parts['pop_indigena_ti'].notna() & (parts['areasum_mun']>0),
    parts['pop_indigena_ti'] * (parts['area_km2'] / parts['areasum_mun']),
    np.nan
)

# fallback: se pop_TI_mun é NaN mas pop_indig existe, alocar pop_indig proporcional à area (ou igualmente)
# parts['pop_alloc_fallback'] = np.where(
#     parts['pop_indigena_ti'].isna() & parts['pop_indigena'].notna() & (parts['areasum_mun']>0),
#     parts['pop_indigena'] * (parts['area_km2'] / parts['areasum_mun']),  # ou usar equal share: parts['pop_indig']/n_tis_mun
#     np.nan
# )

parts['pop_alloc_fallback'] = np.where(
    parts['pop_indigena_ti'].isna() & parts['pop_indigena'].notna(),
    parts['pop_indigena'] / parts['n_tis_mun'],  # n_tis_mun = número de TIs no município
    np.nan
)

# escolher prioridade: pop_alloc > pop_alloc_fallback
parts['pop_part_final'] = parts['pop_alloc'].fillna(parts['pop_alloc_fallback']).fillna(0)

# agregar por TI
ti_est = parts.groupby('ti_id', as_index=False).agg({
    'area_km2':'sum',
    'pop_part_final':'sum'
}).rename(columns={'pop_part_final':'pop_TI_est'})

# join para criar representatividade (usar pop_total do mun mais relevante: por exemplo, pesando pelo maior overlap ou pela parte principal)
# simplificação: pegar mun_id dominante por ti (maior area overlap)
dominant = parts.sort_values(['ti_id','area_km2'], ascending=[True,False]).groupby('ti_id', as_index=False).first()[['ti_id','mun_id','pop_total']]
ti_est = ti_est.merge(dominant, on='ti_id', how='left')
ti_est['representatividade'] = ti_est['pop_TI_est'] / ti_est['pop_total']

# ti_est.to_csv('ti_representatividade.csv', index=False)
# print("Pronto: ti_representatividade.csv")


KeyError: 'n_tis_mun'

In [230]:
parts

,ti_id,mun_id,geometry,area_km2,areasum_mun,pop_total,pop_indigena,pop_indigena_ti,pop_alloc
0,0,245,"POLYGON Z ((5366444.777 6711204.291 -9.425e-5,...",2.973008e+00,20.055441,7418,108,NaN,NaN
1,1,328,"POLYGON Z ((5287320.547 6653618.294 -9.425e-5,...",3.295298e+00,10.334302,1332845,2957,NaN,NaN
2,1,484,MULTIPOLYGON Z (((5287362.967 6653693.217 -9.4...,3.147687e-08,26.363595,224112,959,137.0,1.635714e-07
3,2,328,"POLYGON Z ((5276501.372 6670355.457 -9.425e-5,...",1.460147e-01,10.334302,1332845,2957,NaN,NaN
4,3,41,"POLYGON Z ((5259565.776 6647077.517 -9.425e-5,...",1.131268e+00,174.538833,12225,263,NaN,NaN
...,...,...,...,...,...,...,...,...,...
219,136,174,"POLYGON Z ((5034179.323 6674599.896 -9.425e-5,...",3.355205e+00,3.355205,6413,28,NaN,NaN
220,137,358,"POLYGON Z ((5237681.863 6735262.7 -9.425e-5, 5...",3.103395e-01,0.310339,6879,9,NaN,NaN
221,138,401,"POLYGON Z ((5277077.077 6797473.365 -9.425e-5,...",1.827912e+00,1.827912,21084,26,NaN,NaN
222,139,52,"POLYGON Z ((5375190.288 6818996.437 -9.425e-5,...",8.615140e-01,0.861514,11202,18,NaN,NaN


In [187]:
parts['pop_part_final'].astype(int).sum()

29611

In [167]:
# ti_est.pop_TI_est.astype(int).sum()
ti_est.pop_TI_est.astype(int).sum()

29626

In [120]:
pop['pop_indigena_ti'].astype('Int64').sum() / pop['pop_indigena'].sum()
# * pop['perc_em_ti'].mean()

0.4329400033239156

In [116]:
pop.columns

Index(['CodNivTerr', 'Geocodigo', 'Nome', 'PessInd', 'PessIndEmT',
       'PessIndFor', 'PercPessIn', 'PercPessI0', 'PopResid', 'PopResidEm',
       'PopResidFo', 'PercPessI1', 'PercPessI2', 'PercPessI3', 'geometry',
       'mun_id', 'pop_indigena', 'PopIndEmT', 'pop_indigena_ti', 'pop_total',
       'perc_indigena', 'perc_em_ti'],
      dtype='object')

In [117]:
pop['perc_indigena'].mean()

0.009549988052957831

In [186]:
pop['perc_em_ti'].describe()

count    26.000000
mean      0.746976
std       0.290204
min       0.068404
25%       0.640036
50%       0.898521
75%       0.966987
max       0.997575
Name: perc_em_ti, dtype: float64

In [121]:
pop['pop_indigena'].sum() * pop['perc_em_ti'].mean()

26967.323375998174

In [131]:
parts['pop_part_final'].sum()

29681.0

In [135]:
ti_corr

,ti_id,pop_corrigida_global,pop_corrigida_mun,mun_id,pop_total,IR_global,IR_municipal
0,0,8.430785,0.0,245,7418,0.001137,0.0
1,1,496.529840,0.0,328,1332845,0.000373,0.0
2,2,22.001237,0.0,328,1332845,0.000017,0.0
3,3,0.897655,0.0,41,12225,0.000073,0.0
4,4,1.578967,0.0,245,7418,0.000213,0.0
...,...,...,...,...,...,...,...
136,136,14.744786,0.0,174,6413,0.002299,0.0
137,137,4.739396,0.0,358,6879,0.000689,0.0
138,138,13.691587,0.0,401,21084,0.000649,0.0
139,139,9.478791,0.0,52,11202,0.000846,0.0


In [144]:
ti_corr['IR_municipal_%'] = ti_corr.IR_municipal * 100

In [146]:
ti_corr['IR_municipal_%'].describe()

count         141.0
mean       3.247594
std       16.241949
min             0.0
25%             0.0
50%             0.0
75%        0.001733
max      167.930453
Name: IR_municipal_%, dtype: Float64

In [152]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
ti_corr['IR_global_norm'] = scaler.fit_transform(ti_corr['IR_global'].values.reshape(-1, 1) )

In [157]:
ti_corr['IR_global_norm'].mean()

0.022569930352176856

In [199]:
import pandas as pd
import numpy as np

# Supondo que 'parts' já exista e tenha colunas: ti_id, mun_id, pop_part_final
# E df_mun tem: mun_id, pop_total, pop_indig_total

# Merge população indígena total do município
# parts = parts.merge(pop[['mun_id','pop_indigena']], on='mun_id', how='left')

# Evitar divisão por zero ou nulos
parts['pop_indigena'] = parts['pop_indigena'].fillna(0)
parts['pop_part_final'] = parts['pop_part_final'].fillna(0)

parts['IR_municipal_corrigido'] = np.where(
    parts['pop_indigena_ti'] > 0,
    parts['pop_part_final'] / parts['pop_indigena_ti'],  # usa valor da TI quando existe
    np.where(parts['pop_indigena'] > 0,
             parts['pop_part_final'] / parts['pop_indigena'],  # fallback municipal
             0)
)

# Se TI cruza múltiplos municípios, agregar por TI usando soma
ti_corr = parts.groupby('ti_id', as_index=False).agg({
    'pop_part_final':'sum',
    'IR_municipal_corrigido':'sum'
})

# Calcular IR global também, se quiser
ti_corr['IR_global'] = ti_corr['pop_part_final'] / parts.groupby('ti_id')['pop_part_final'].sum().max()

# Exportar
ti_corr.to_csv('ti_representatividade_municipal_corrigida.csv', index=False)
print("Arquivo salvo: ti_representatividade_municipal_corrigida.csv")


Arquivo salvo: ti_representatividade_municipal_corrigida.csv


In [216]:
parts['pop_part_final'].max()

3318.5508585696484

In [193]:
ti_est['representatividade'].describe()

count    1.410000e+02
mean     3.790254e-02
std      1.633144e-01
min      7.735177e-07
25%      1.520160e-04
50%      1.308330e-03
75%      5.823340e-03
max      1.679305e+00
Name: representatividade, dtype: float64

In [203]:
ti_corr['IR_municipal_corrigido'].describe()

count    141.000000
mean       0.801418
std        0.893042
min        0.000322
25%        0.053261
50%        0.695967
75%        1.000000
max        4.311792
Name: IR_municipal_corrigido, dtype: float64

In [202]:
ti_corr['IR_global'].describe()

count    141.000000
mean       0.038504
std        0.104694
min        0.000011
25%        0.002708
50%        0.010426
75%        0.023736
max        1.000000
Name: IR_global, dtype: float64

In [206]:
ti_corr.pop_part_final.astype(int).sum()

29626

In [205]:
ti_est.pop_TI_est.astype(int).sum()

29626

In [222]:
parts.pop_alloc_fallback.sum()

14051.0

In [231]:
import pandas as pd
import numpy as np

# Supondo que você tenha:
# parts: dataframe com colunas ['ti_id', 'mun_id', 'area_km2', 'areasum_mun', 'pop_indigena_ti', 'pop_indigena', 'n_tis_mun']

# 1️⃣ Alocação principal: usar pop_indigena_ti quando disponível
parts['pop_alloc'] = np.where(
    parts['pop_indigena_ti'].notna() & (parts['areasum_mun']>0),
    parts['pop_indigena_ti'] * (parts['area_km2'] / parts['areasum_mun']),
    np.nan
)

# 2️⃣ Fallback: se pop_indigena_ti é NaN, usar pop_indigena proporcional à área
parts['pop_alloc_fallback'] = np.where(
    parts['pop_indigena_ti'].isna() & parts['pop_indigena'].notna() & (parts['areasum_mun']>0),
    parts['pop_indigena'] * (parts['area_km2'] / parts['areasum_mun']),
    np.nan
)

# 3️⃣ Combinar alocação principal + fallback
parts['pop_part_final'] = parts['pop_alloc'].combine_first(parts['pop_alloc_fallback'])

# 4️⃣ Calcular IR_municipal com fallback robusto
parts['IR_municipal_corrigido'] = np.where(
    parts['pop_indigena_ti'].notna() & (parts['pop_indigena_ti']>0),
    parts['pop_part_final'] / parts['pop_indigena_ti'],  # usar dado da TI quando disponível
    np.where(
        parts['pop_indigena'].notna() & (parts['pop_indigena']>0),
        parts['pop_part_final'] / parts['pop_indigena'],   # fallback: população indígena do município
        0
    )
)

# 5️⃣ Calcular IR_global (normalização pelo máximo da população estimada)
ti_max_pop = parts.groupby('ti_id')['pop_part_final'].sum().max()
ti_grouped = parts.groupby('ti_id', as_index=False).agg({
    'pop_part_final':'sum'
})
ti_grouped['IR_global'] = ti_grouped['pop_part_final'] / ti_max_pop

# 6️⃣ Merge IR_municipal médio por TI (caso TI cruza múltiplos municípios)
IR_mun_avg = parts.groupby('ti_id')['IR_municipal_corrigido'].mean().reset_index()
ti_grouped = ti_grouped.merge(IR_mun_avg, on='ti_id', how='left')
ti_grouped.rename(columns={'IR_municipal_corrigido':'IR_municipal_corrigido_avg'}, inplace=True)

# 7️⃣ Resultado final pronto para análise de vulnerabilidade
ti_grouped.to_csv('ti_representatividade_final.csv', index=False)
print("Arquivo salvo: ti_representatividade_final.csv")


Arquivo salvo: ti_representatividade_final.csv


In [235]:
ti_grouped

,ti_id,pop_part_final,IR_global,IR_municipal_corrigido_avg
0,0,16.009861,0.002928,0.148239
1,1,942.898411,0.172471,0.159435
2,2,41.779829,0.007642,0.014129
3,3,1.704626,0.000312,0.006481
4,4,2.998421,0.000548,0.027763
...,...,...,...,...
136,136,28.000000,0.005122,1.000000
137,137,9.000000,0.001646,1.000000
138,138,26.000000,0.004756,1.000000
139,139,18.000000,0.003292,1.000000
